In [14]:
import pandas as pd

In [15]:
df = pd.read_excel("fpflagc1927.xlsx", skiprows=1)

In [16]:
print(df.columns)

Index(['PSO No', 'PSO Date', 'Agency', 'Partno', 'Order', 'Cancl', 'Supply',
       'RtnQty'],
      dtype='object')


In [17]:
# Create the new column logic
df["Rtn=Ord"] = df.apply(
    lambda x: "OUT" if x["RtnQty"] == x["Order"] else "OK",
    axis=1
)

# Insert the column right after 'RtnQty'
rtnqty_index = df.columns.get_loc("RtnQty")
df.insert(rtnqty_index + 1, "Rtn=Ord", df.pop("Rtn=Ord"))

In [18]:
# Create the new column logic
df["Rtn+Cncl=Ord"] = df.apply(
    lambda x: "OUT" if x["RtnQty"] + x["Cancl"] == x["Order"] else "OK",
    axis=1
)

# Insert the column right after 'Rtn=Ord'
rtnqty_index = df.columns.get_loc("Rtn=Ord")
df.insert(rtnqty_index + 1, "Rtn+Cncl=Ord", df.pop("Rtn+Cncl=Ord"))

In [19]:
# Create the new column logic
df["Ord=0"] = df.apply(
    lambda x: "OUT" if x["Order"] == 0 else "OK",
    axis=1
)

# Insert the column right after 'Rtn=Ord'
rtnqty_index = df.columns.get_loc("Rtn+Cncl=Ord")
df.insert(rtnqty_index + 1, "Ord=0", df.pop("Ord=0"))

In [20]:
# Create "Doc Type" from PSO No (5th–6th characters)
df["Doc Type"] = (
    df["PSO No"]
    .astype(str)
    .str[4:6]   # Python is 0-based → 5th char = index 4
)

pso_index = df.columns.get_loc("Ord=0")
df.insert(pso_index + 1, "Doc Type", df.pop("Doc Type"))

In [21]:
# --- Create Release column ---
df["Release"] = df["PSO No"].astype(str).str[-1].apply(
    lambda x: "No" if x.isdigit() else "Release"
)

# Insert after "Doc Type"
doc_index = df.columns.get_loc("Doc Type")
df.insert(doc_index + 1, "Release", df.pop("Release"))

In [22]:
df2 = pd.read_excel("brc 27 agc 19.xlsx")

In [23]:
# --- Normalize part numbers (CRITICAL) ---
df["Partno"] = df["Partno"].astype(str).str.strip().str.upper()
df2["p/n"] = df2["p/n"].astype(str).str.strip().str.upper()

# --- Select only needed columns from df2 ---
df2_merge = df2[["p/n", "FD_final", "RC"]].drop_duplicates()

# --- Merge ---
df = df.merge(
    df2_merge,
    left_on="Partno",
    right_on="p/n",
    how="left"
)

# --- Drop duplicate key column ---
df.drop(columns=["p/n"], inplace=True)

In [24]:
print(df)

             PSO No   PSO Date  Agency   Partno  Order  Cancl  Supply  RtnQty  \
0      2427SA29506D 2025-04-08      19  3282141      6      0       4       0   
1      2427SA29544A 2025-01-20      19  3920780      2      0       2       0   
2       2427SA29598 2024-12-17      19  5690194      0      0       0       2   
3      2427SA29598A 2024-12-17      19  5333477      0      0       0       1   
4      2427SA29604A 2025-01-08      19  4937767      1      0       1       0   
...             ...        ...     ...      ...    ...    ...     ...     ...   
13777   2527SW10020 2025-12-29      19  3090769      1      0       1       0   
13778   2527SW10020 2025-12-29      19  3408324      1      0       1       0   
13779   2527SW10020 2025-12-29      19  3803615      1      0       1       0   
13780   2527SW10020 2025-12-29      19  3803780      1      0       1       0   
13781   2527SW10020 2025-12-29      19  3050624      1      0       1       0   

      Rtn=Ord Rtn+Cncl=Ord 

In [25]:
df.to_excel(f"fpfldet agc19 brc27.xlsx", index=False)